# Bloomberg training-data pull: Fuzzy PPO execution (NSE), research tier

**What this produces.** It fetches the raw Bloomberg data that the training pipeline reads (`hftf data build-features`
→ `features_1m_v2` → `hftf-sim` episodes), in the on-disk layout `hftf_data.loaders` expects. The goal is to take
the data from the *pilot* tier (3 stocks) to the *research* tier in `docs/DATA_READINESS.md`:
≥ 50 NSE equities × ≥ 90 sessions of 1-min TRADE/BID/ASK bars, no session gaps, NIFTY/BANKNIFTY/VIX context bars,
and ≥ 10 equities with ≥ 40 full tick days.

**How to run.** Open the Terminal and log in. Then **Kernel → Restart & Run All**. The first cell shows `MODE`.
- `MODE = "plan"` contacts nothing. It prints which sessions and tick days are missing and how many requests a pull
  would make. It also works on the Mac against `data/is2/bbg_data`.
- `MODE = "pull"` does the pull. It is **incremental and gap-driven**: it asks Bloomberg only for sessions that are
  missing or partial on disk, so re-running costs nothing for data you already have. Run it **every week**. Ticks
  roll off after ~1 month and bars after ~140 days, and R05 needs at least two tick pulls about 4 weeks apart.

**Read-only.** Nothing here places orders. If Bloomberg reports a daily/monthly data limit, the pull stops at once,
keeps everything already fetched, and resumes from the gaps on the next run.

### What changed compared with the previous pull (`LIMIT = 3`, 2026-10-21)
| Problem in the previous pull | Fix here |
|---|---|
| Nifty weights come back as the N/A sentinel, so "top N by weight" became **alphabetical** (ADE, ADSEZ, APHS) | Universe = all current Nifty-50 members + a small Nifty Next 50 buffer ranked by market cap; tick names ranked by 60-day median turnover |
| Starter seeds were `IN` (composite) tickers while members resolve as `IS` (NSE), so a full pull would fetch ~28 names twice on the wrong book | Every equity is normalised to the NSE `IS` ticker; seeds are a fallback only |
| Bar chunk file names depended on the run date, so every re-run re-downloaded all bars | Planning is per **session**: only missing/partial sessions are requested, and file names never collide |
| 2026-09-10 bars missing (needed manual file deletion); 2026-10-21 partial and never refreshed | Gaps and partial days (last bar < 15:20 IST) are found automatically and re-requested |
| Tick universe not persisted, but R05 needs the **same** 10+ names across pulls | `reference/tick_universe.json` is fixed on first use and always keeps names that already have ticks |
| The probe ran on an unresolvable ticker (`RIL IN Equity`), so lookbacks and the timezone were never checked | Probe on a resolved member; binary-searches the real bar/tick history depth; **aborts** if bar times are not UTC |
| 625 futures contracts + 936 options + invalid fields gave 9.7k errors and wasted quota | Futures = NZ1/NZ2/AF1/AF2 generics only, no options (out of scope for v1); invalid fields removed |
| Daily history parts were never extended | Daily parts are appended incrementally; a split/adjustment change forces a full re-fetch of that name |
| The session being traded "today" could be stored as a partial day | Only sessions finished before 16:10 IST are requested |
| A crash mid-write could leave a corrupt parquet | Atomic writes (temp file + rename) |

The cleaning fixes F1-F9 in `docs/DATA_AUDIT.md` (tick condition codes, crossed quotes, sentinels, pre-open zeros,
tick size) stay in `hftf-data`. This notebook stores raw responses unchanged, as before.

## 1. Settings
Edit only this cell. The defaults are the research-tier pull.

In [ ]:
from pathlib import Path

MODE = "pull"  # "plan" = no Bloomberg calls, print the gap report | "pull" = fetch the gaps
HOST, PORT = "localhost", 8194

# Where the raw pull lives. None = `data/is2/bbg_data` when run inside the repo, else `./bbg_data`.
# On the Bloomberg PC, keep the SAME folder between weekly runs: it is what makes the pull incremental.
OUT = None

# ---- universe ------------------------------------------------------------------------------------
MEMBER_INDEX = "NIFTY Index"  # all current members are pulled intraday
BUFFER_INDEX = (
    "NIFTYJR Index"  # Nifty Next 50. VERIFY in the Terminal (`NIFTY NEXT 50 <INDEX>`); skipped if unresolved
)
BUFFER_N = 5  # extra liquid names so one bad member cannot drop R01 below 50
TICK_UNIVERSE_N = 12  # R05 needs 10 names with >= 40 full tick days; 2 spare
# Fallback only if the membership request returns nothing (NSE `IS` tickers).
FALLBACK_EQUITIES = [
    "RELIANCE IS Equity",
    "HDFCB IS Equity",
    "ICICIBC IS Equity",
    "INFO IS Equity",
    "TCS IS Equity",
    "BHARTI IS Equity",
    "LT IS Equity",
    "ITC IS Equity",
    "SBIN IS Equity",
    "AXSB IS Equity",
    "KMB IS Equity",
    "HUVR IS Equity",
    "BAF IS Equity",
    "MSIL IS Equity",
    "MM IS Equity",
    "SUNP IS Equity",
    "HCLT IS Equity",
    "NTPC IS Equity",
    "TTAN IS Equity",
    "PWGR IS Equity",
]
CONTEXT_INDICES = ["NIFTY Index", "NSEBANK Index", "INVIXN Index"]  # intraday TRADE bars (R04)
DAILY_ONLY_INDICES = [
    "SENSEX Index",
    "NSEIT Index",
    "NSEAUTO Index",
    "NSEFMCG Index",
    "NSEPHRM Index",
    "NSEMET Index",
    "NSEFIN Index",
    "NSEMCAP Index",
    "NSESMCP Index",
    "BSE500 Index",
    "SPX Index",
    "NDX Index",
    "VIX Index",
    "SX5E Index",
    "NKY Index",
    "HSI Index",
]
FX_INTRADAY = ["USDINR Curncy", "EURINR Curncy"]  # BID/ASK bars
FX_DAILY_ONLY = ["DXY Curncy", "USDINR1M Curncy", "USDINRV1M Curncy"]  # DXY has no intraday entitlement
RATES = ["GIND10YR Index", "GIND5YR Index", "GIND2YR Index", "USGG10YR Index", "USGG2YR Index"]
COMMODITIES = ["CO1 Comdty", "CL1 Comdty", "GC1 Comdty"]
FUTURES_INTRADAY = ["NZ1 Index", "NZ2 Index", "AF1 Index"]  # Nifty fut front/next, BankNifty front
FUTURES_DAILY_ONLY = ["AF2 Index"]

# ---- history windows -----------------------------------------------------------------------------
DAILY_YEARS = 15
DEFAULT_BAR_LOOKBACK_DAYS = 140  # used only when the probe cannot measure it
DEFAULT_TICK_LOOKBACK_DAYS = 30
BAR_CHUNK_SESSIONS = 5  # sessions per IntradayBarRequest
SESSION_DONE_IST = "16:10"  # today's session counts as finished after this IST time
FULL_DAY_LAST_IST = "15:20"  # a day is complete if its data reaches this IST time (= readiness rule)
FULL_TICK_DAY_FIRST_IST = "09:16"  # ... and, for ticks, starts by this time (edge-of-window days start late)
RETRY_EMPTY = False  # True = re-ask sessions Bloomberg previously returned empty for

MAKE_BUNDLE = True  # zip the files written by this run (not needed when OUT is the repo folder)

## 2. Environment check
`blpapi` comes from Bloomberg's package index. Uncomment the install line once per PC if the import fails.

In [ ]:
import importlib.util
import sys

# !{sys.executable} -m pip install --user pandas numpy pyarrow tzdata
# !{sys.executable} -m pip install --user --index-url=https://blpapi.bloomberg.com/repository/releases/python/simple/ blpapi  # noqa: E501
import numpy as np
import pandas as pd

try:
    import blpapi  # type: ignore

    BLPAPI_OK, BLPAPI_ERR = True, ""
except Exception as _e:  # plan mode works without it
    blpapi, BLPAPI_OK, BLPAPI_ERR = None, False, repr(_e)
for _m in ("pyarrow",):
    if importlib.util.find_spec(_m) is None:
        raise ImportError(f"{_m} is required (pip install --user {_m})")
print(
    f"python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__} | "
    f"blpapi {blpapi.__version__ if BLPAPI_OK else 'NOT AVAILABLE: ' + BLPAPI_ERR}"
)
if MODE == "pull" and not BLPAPI_OK:
    raise RuntimeError("MODE='pull' needs blpapi in this kernel. Install it (cell above) or set MODE='plan'.")

## 3. Library: utilities, calendar, on-disk state
Everything below is plumbing; the run cell is at the end. Timestamps are stored in UTC. IST (UTC+05:30, no DST)
is used only to reason about the NSE session, so a fixed offset is exact.

In [ ]:
import contextlib
import json
import math
import os
import re
import time
import traceback
import zipfile
from datetime import UTC, date, datetime, timedelta


def _default_out() -> Path:
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / "data" / "is2").is_dir() and (base / "packages" / "hftf-data").is_dir():
            return base / "data" / "is2" / "bbg_data"
    return Path("bbg_data")


OUT = Path(OUT) if OUT else _default_out()
IN_REPO = OUT.resolve().parent.name == "is2"
RUN_STARTED = datetime.now(UTC)
RUN_TAG = RUN_STARTED.strftime("%Y%m%d%H%M")
IST_OFFSET = timedelta(hours=5, minutes=30)
TODAY = (RUN_STARTED + IST_OFFSET).date()
DAILY_START = date(TODAY.year - DAILY_YEARS, TODAY.month, min(TODAY.day, 28))
MAX_FIELDS_PER_HIST_REQ = 25
REQUEST_SLEEP = 0.05
MAX_RETRIES = 3
REFDATA, APIFLDS = "//blp/refdata", "//blp/apiflds"
TICK_EVENTS = ["TRADE", "BID", "ASK"]  # the event set hftf_data.ticks is validated against
TICK_WINDOW_UTC = ((3, 30), (10, 5))  # 09:00-15:35 IST: pre-open, continuous session, close
# Weekday NSE holidays; used only for dates beyond the NIFTY daily history (normally none).
KNOWN_NSE_HOLIDAYS = {date(2026, 6, 26), date(2026, 9, 14), date(2026, 10, 2), date(2026, 10, 20)}
STATS = {"requests": 0, "written_files": 0}


def log(msg: str) -> None:
    print(f"[{datetime.now():%H:%M:%S}] {msg}", flush=True)


def hhmm(s: str):
    return datetime.strptime(s, "%H:%M").time()


def dir_name(sec: str) -> str:
    """'ADE IS Equity' -> 'ADE_IS_Equity'. Same as hftf_data.paths.safe_name (the loaders' folder names)."""
    return "_".join(str(sec).replace("/", " ").split())


def chunks(seq, n):
    seq = list(seq)
    for i in range(0, len(seq), n):
        yield seq[i : i + n]


def utc_dt(d: date, end_of_day: bool = False) -> datetime:
    """Naive datetime, which Bloomberg interprets as UTC."""
    return datetime(d.year, d.month, d.day, 23, 59, 59) if end_of_day else datetime(d.year, d.month, d.day)


def to_ist_naive(s) -> pd.Series:
    return pd.to_datetime(pd.Series(s), utc=True).dt.tz_convert(None) + IST_OFFSET


def last_complete_session_day() -> date:
    now_ist = (datetime.now(UTC) + IST_OFFSET).replace(tzinfo=None)
    return now_ist.date() if now_ist.time() >= hhmm(SESSION_DONE_IST) else now_ist.date() - timedelta(days=1)


def atomic_write(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")  # '*.parquet' globs in the loaders never see it
    df.to_parquet(tmp, index=False)
    os.replace(tmp, path)
    STATS["written_files"] += 1


def save_json(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, default=str))
    os.replace(tmp, path)


def load_json(path: Path, default=None):
    return json.loads(path.read_text()) if path.exists() else default


def log_error(kind: str, **kw) -> None:
    if MODE != "pull":
        return
    p = OUT / "reports" / "errors.jsonl"
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, "a", encoding="utf-8") as f:
        f.write(json.dumps({"ts": datetime.now(UTC).isoformat(), "kind": kind, **kw}, default=str) + "\n")


def _stringify_nested(df: pd.DataFrame) -> pd.DataFrame:
    """Lists/dicts -> JSON strings and mixed object columns -> str, so parquet can store them."""
    for c in df.columns:
        if df[c].dtype != object:
            continue
        vals = df[c].dropna()
        if vals.map(lambda v: isinstance(v, (list, dict))).any():
            df[c] = df[c].map(lambda v: json.dumps(v, default=str) if isinstance(v, (list, dict)) else v)
            vals = df[c].dropna()
        numeric = vals.map(lambda v: isinstance(v, (int, float, np.number, bool))).all()
        temporal = vals.map(lambda v: isinstance(v, (pd.Timestamp, datetime, date))).all()
        if len(vals) and not numeric and not temporal:
            df[c] = df[c].map(
                lambda v: None if v is None or (isinstance(v, float) and math.isnan(v)) else str(v)
            )
    return df


def coerce_numeric(
    df: pd.DataFrame, keep=("security", "date", "time", "class", "event", "type")
) -> pd.DataFrame:
    for c in df.columns:
        if c in keep or df[c].dtype != object:
            continue
        num = pd.to_numeric(df[c], errors="coerce")
        if num.notna().sum() == df[c].notna().sum():
            df[c] = num
        else:
            df[c] = df[c].map(
                lambda v: None if v is None or (isinstance(v, float) and math.isnan(v)) else str(v)
            )
    return df


class Ledger:
    """Sessions Bloomberg answered with nothing (or a final partial day): not re-asked every run."""

    def __init__(self, path: Path):
        self.path = path
        self.data = load_json(path, {}) or {}

    def get(self, key: str, d: date):
        return None if RETRY_EMPTY else self.data.get(key, {}).get(str(d))

    def mark(self, key: str, d: date, reason: str) -> None:
        self.data.setdefault(key, {})[str(d)] = reason

    def save(self) -> None:
        if MODE == "pull":
            save_json(self.data, self.path)


# ---- NSE session calendar ------------------------------------------------------------------------
def _nifty_daily() -> pd.DataFrame:
    for p in (OUT / "daily" / "_parts" / "index__NIFTY_Index.parquet", OUT / "daily" / "daily_index.parquet"):
        if p.exists():
            df = pd.read_parquet(p, columns=["security", "date", "PX_LAST"])
            return df[(df["security"] == "NIFTY Index") & df["PX_LAST"].notna()]
    return pd.DataFrame(columns=["security", "date", "PX_LAST"])


def nse_sessions(start: date, end: date) -> list:
    """NSE sessions in [start, end]: NIFTY daily history, then weekdays after it (minus known holidays)."""
    nd = _nifty_daily()
    days = set(pd.to_datetime(nd["date"]).dt.date) if len(nd) else set()
    covered = max(days) if days else start - timedelta(days=1)
    d = covered + timedelta(days=1)
    while d <= end:
        if d.weekday() < 5 and d not in KNOWN_NSE_HOLIDAYS:
            days.add(d)
        d += timedelta(days=1)
    return sorted(x for x in days if start <= x <= end)


# ---- what is already on disk ---------------------------------------------------------------------
def bar_folder(event: str, sec: str) -> Path:
    return OUT / "intraday_bars" / "1m" / event / dir_name(sec)


def tick_folder(sec: str) -> Path:
    return OUT / "ticks" / dir_name(sec)


def day_extents(files) -> dict:
    """{IST date: (first IST time, last IST time)} over parquet files, reading only the `time` column."""
    out: dict = {}
    for f in files:
        try:
            t = pd.read_parquet(f, columns=["time"])["time"]
        except Exception as e:  # unreadable file: treat its days as missing so they are re-fetched
            log(f"  unreadable {f}: {e!r}")
            continue
        if not len(t):
            continue
        ist = to_ist_naive(t)
        g = ist.groupby(ist.dt.date).agg(["min", "max"])
        for d, (lo, hi) in g.iterrows():
            a, b = out.get(d, (lo, hi))
            out[d] = (min(a, lo), max(b, hi))
    return {d: (lo.time(), hi.time()) for d, (lo, hi) in out.items()}


def equities_with_bars() -> list:
    root = OUT / "intraday_bars" / "1m" / "TRADE"
    if not root.exists():
        return []
    return sorted(
        d.name.replace("_", " ")
        for d in root.iterdir()
        if d.is_dir() and d.name.endswith("_Equity") and any(d.glob("*.parquet"))
    )


def equities_with_ticks() -> list:
    root = OUT / "ticks"
    if not root.exists():
        return []
    return sorted(d.name.replace("_", " ") for d in root.iterdir() if d.is_dir() and any(d.glob("*.parquet")))

## 4. Library: Bloomberg session and request layer
Same request layer as the previous collector, which worked on this Terminal: correlation-matched replies, an idle
timeout that never kills a response that is still streaming, retries on connection hiccups, and a hard stop on
data-limit messages.

In [ ]:
SESSION = None
_OPEN_SERVICES: set = set()
_CID = [0]


class BBGRequestFailure(Exception):  # noqa: N818 (name kept from the collector)
    """Bloomberg answered RequestFailure (bad request, entitlement). Not retried."""


class BBGLimitReached(Exception):  # noqa: N818
    """Bloomberg reported a daily/monthly data limit. The pull stops so the shared terminal is spared."""


def connect():
    global SESSION
    if SESSION is not None:
        return SESSION
    if not BLPAPI_OK:
        raise RuntimeError(f"blpapi not importable: {BLPAPI_ERR}")
    o = blpapi.SessionOptions()
    o.setServerHost(HOST)
    o.setServerPort(PORT)
    s = blpapi.Session(o)
    if not s.start():
        raise RuntimeError(
            f"Could not start a Bloomberg session at {HOST}:{PORT}. Is the Terminal logged in?"
        )
    SESSION = s
    open_service(REFDATA)
    return s


def close():
    global SESSION
    if SESSION is not None:
        with contextlib.suppress(Exception):
            SESSION.stop()
    SESSION = None
    _OPEN_SERVICES.clear()


def open_service(name: str):
    if SESSION is None:
        connect()
    if name not in _OPEN_SERVICES:
        if not SESSION.openService(name):
            raise RuntimeError(f"Could not open service {name}")
        _OPEN_SERVICES.add(name)
    return SESSION.getService(name)


def el_to_py(el):
    try:
        if el.isNull():
            return None
    except Exception:
        pass
    try:
        if el.isArray():
            out = []
            for i in range(el.numValues()):
                try:
                    out.append(el_to_py(el.getValueAsElement(i)))
                except Exception:
                    out.append(el.getValue(i))
            return out
        if el.isComplexType():
            return {str(el.getElement(i).name()): el_to_py(el.getElement(i)) for i in range(el.numElements())}
        return el.getValue()
    except Exception:
        return str(el)


_TOP_KEYS = {"securityData", "barData", "tickData", "fieldData", "responseError", "reason", "categories"}


def msg_to_dict(msg) -> dict:
    mtype = str(msg.messageType())
    d = None
    if hasattr(msg, "toPy"):  # fast C-level conversion (blpapi >= 3.20)
        try:
            d = msg.toPy()
            if isinstance(d, dict) and len(d) == 1 and mtype in d and isinstance(d[mtype], dict):
                d = d[mtype]
            if not (isinstance(d, dict) and (set(d) & _TOP_KEYS or mtype == "RequestFailure")):
                d = None
        except Exception:
            d = None
    if d is None:
        d = el_to_py(msg.asElement())
    d = dict(d) if isinstance(d, dict) else {"value": d}
    d["_type"] = mtype
    return d


def _cid_value(c):
    try:
        return c.value()
    except Exception:
        return c


def _is_mine(msg, cid) -> bool:
    try:
        ids = msg.correlationIds()
    except Exception:
        try:
            ids = [msg.correlationId()]
        except Exception:
            return True
    want = _cid_value(cid)
    return any(_cid_value(c) == want for c in ids)


def _limit_hit(d: dict):
    errs = [d["responseError"]] if isinstance(d.get("responseError"), dict) else []
    sds = d.get("securityData")
    for sd in sds if isinstance(sds, list) else [sds] if isinstance(sds, dict) else []:
        if isinstance(sd, dict) and isinstance(sd.get("securityError"), dict):
            errs.append(sd["securityError"])
    for e in errs:
        cat, sub, msg = (str(e.get(k, "")).upper() for k in ("category", "subcategory", "message"))
        if (
            cat == "LIMIT"
            or "LIMIT_REACHED" in sub
            or "DAILY LIMIT" in msg
            or "MONTHLY LIMIT" in msg
            or "DAILY_CAPACITY" in sub
        ):
            return json.dumps(e, default=str)[:400]
    return None


def _send_once(req, timeout: float) -> list:
    _CID[0] += 1
    cid = blpapi.CorrelationId(_CID[0])
    SESSION.sendRequest(req, correlationId=cid)
    STATS["requests"] += 1
    out, last = [], time.time()
    req_status = getattr(blpapi.Event, "REQUEST_STATUS", None)
    while True:
        ev = SESSION.nextEvent(500)
        et = ev.eventType()
        if et in (blpapi.Event.PARTIAL_RESPONSE, blpapi.Event.RESPONSE) or (
            req_status is not None and et == req_status
        ):
            mine = False
            for msg in ev:
                if not _is_mine(msg, cid):
                    continue
                mine = True
                d = msg_to_dict(msg)
                if d.get("_type") == "RequestFailure":
                    reason = json.dumps(d, default=str)[:600]
                    if "LIMIT" in reason.upper() and "REACHED" in reason.upper():
                        raise BBGLimitReached(reason)
                    raise BBGRequestFailure(reason)
                if req_status is not None and et == req_status:
                    continue
                lim = _limit_hit(d)
                if lim:
                    raise BBGLimitReached(lim)
                out.append(d)
            if mine:
                last = time.time()
                if et == blpapi.Event.RESPONSE:
                    return out
        elif et == blpapi.Event.SESSION_STATUS:
            for msg in ev:
                if str(msg.messageType()) in ("SessionTerminated", "SessionConnectionDown"):
                    raise ConnectionError(str(msg))
        if time.time() - last > timeout:
            with contextlib.suppress(Exception):
                SESSION.cancel(cid)
            raise TimeoutError(f"no data for {timeout}s")


def send(make_request, timeout: float = 180) -> list:
    last = None
    for attempt in range(MAX_RETRIES):
        try:
            res = _send_once(make_request(), timeout)
            time.sleep(REQUEST_SLEEP)
            return res
        except (BBGRequestFailure, BBGLimitReached):
            raise
        except Exception as e:
            last = e
            log(f"  retry {attempt + 1}/{MAX_RETRIES} after {e!r}")
            if isinstance(e, ConnectionError):
                close()
                try:
                    connect()
                except Exception as ce:
                    last = ce
            time.sleep(2**attempt)
    raise last  # type: ignore[misc]


# ---- parsers (message dicts -> rows) -------------------------------------------------------------
def _err_rows(sec, sd: dict, errs: list) -> None:
    if sd.get("securityError"):
        errs.append(
            {"security": sec, "type": "securityError", "detail": json.dumps(sd["securityError"], default=str)}
        )
    for fe in sd.get("fieldExceptions") or []:
        info = fe.get("errorInfo") or {}
        errs.append(
            {
                "security": sec,
                "type": "fieldException",
                "field": fe.get("fieldId"),
                "detail": info.get("message") or json.dumps(info, default=str),
            }
        )


def parse_reference(msgs):
    rows, bulk, errs = [], [], []
    for m in msgs:
        sds = m.get("securityData") or []
        for sd in sds if isinstance(sds, list) else [sds]:
            sec = sd.get("security")
            _err_rows(sec, sd, errs)
            row = {"security": sec}
            for k, v in (sd.get("fieldData") or {}).items():
                if isinstance(v, list):
                    for i, item in enumerate(v):
                        rec = {"security": sec, "field": k, "row": i}
                        rec.update(item if isinstance(item, dict) else {"value": item})
                        bulk.append(rec)
                else:
                    row[k] = v
            rows.append(row)
    return rows, bulk, errs


def parse_historical(msgs):
    rows, errs = [], []
    for m in msgs:
        sds = m.get("securityData") or []
        for sd in sds if isinstance(sds, list) else [sds]:
            sec = sd.get("security")
            _err_rows(sec, sd, errs)
            for item in sd.get("fieldData") or []:
                rows.append({"security": sec, **item})
    return rows, errs


def parse_bars(msgs, security: str, event: str):
    rows, errs = [], []
    for m in msgs:
        if m.get("responseError"):
            errs.append(
                {
                    "security": security,
                    "type": "responseError",
                    "detail": json.dumps(m["responseError"], default=str),
                }
            )
        for bar in (m.get("barData") or {}).get("barTickData") or []:
            rows.append({**bar, "security": security, "event": event})
    return rows, errs


def parse_ticks(msgs, security: str):
    rows, errs = [], []
    for m in msgs:
        if m.get("responseError"):
            errs.append(
                {
                    "security": security,
                    "type": "responseError",
                    "detail": json.dumps(m["responseError"], default=str),
                }
            )
        for t in (m.get("tickData") or {}).get("tickData") or []:
            rows.append({**t, "security": security})
    return rows, errs


def bulk_member(rec: dict):
    """Ticker + weight from an INDX_MWEIGHT / INDX_MWEIGHT_HIST / INDX_MEMBERS bulk row."""
    mem = wt = None
    for k, v in rec.items():
        kl = str(k).lower()
        if mem is None and ("member" in kl or "ticker" in kl) and isinstance(v, str) and v.strip():
            mem = v.strip()
        elif wt is None and "weight" in kl:
            wt = v
    return mem, wt


def nse_equity(s: str) -> str:
    """'RELIANCE IS' / 'RELIANCE IN Equity' -> 'RELIANCE IS Equity' (NSE book, not the Indian composite)."""
    s = " ".join(str(s).split())
    if not s.endswith(("Equity", "Index", "Comdty", "Curncy", "Corp", "Govt")):
        s += " Equity"
    return re.sub(r" IN Equity$", " IS Equity", s)


# ---- request wrappers ----------------------------------------------------------------------------
def ref_request(securities, fields, overrides=None):
    def mk():
        r = open_service(REFDATA).createRequest("ReferenceDataRequest")
        for s in securities:
            r.getElement("securities").appendValue(s)
        for f in fields:
            r.getElement("fields").appendValue(f)
        for k, v in (overrides or {}).items():
            o = r.getElement("overrides").appendElement()
            o.setElement("fieldId", k)
            o.setElement("value", str(v))
        return r

    return parse_reference(send(mk))


def hist_request(sec, fields, start: date, end: date, adjust=True) -> pd.DataFrame:
    """Daily history for ONE security; fields chunked to Bloomberg's per-request limit, merged on date."""
    merged = None
    for fchunk in chunks(fields, MAX_FIELDS_PER_HIST_REQ):

        def mk(fchunk=fchunk):
            r = open_service(REFDATA).createRequest("HistoricalDataRequest")
            r.getElement("securities").appendValue(sec)
            for f in fchunk:
                r.getElement("fields").appendValue(f)
            r.set("startDate", start.strftime("%Y%m%d"))
            r.set("endDate", end.strftime("%Y%m%d"))
            r.set("periodicitySelection", "DAILY")
            r.set("nonTradingDayFillOption", "ACTIVE_DAYS_ONLY")
            r.set("adjustmentFollowDPDF", False)
            r.set("adjustmentSplit", bool(adjust))
            r.set("adjustmentAbnormal", bool(adjust))
            r.set("adjustmentNormal", False)
            return r

        try:
            rows, errs = parse_historical(send(mk, timeout=240))
        except BBGRequestFailure as e:
            log_error("hist_request_failure", security=sec, detail=str(e))
            continue
        for e in errs:
            log_error("hist_error", **e)
        df = pd.DataFrame(rows)
        if df.empty or "date" not in df.columns:
            continue
        df["date"] = pd.to_datetime(df["date"])
        df = coerce_numeric(df.drop(columns=["security"]))
        merged = df if merged is None else merged.merge(df, on="date", how="outer")
    if merged is None:
        return pd.DataFrame()
    merged.insert(0, "security", sec)
    return merged.sort_values("date").reset_index(drop=True)


def bars_request(sec: str, event: str, start_dt: datetime, end_dt: datetime) -> pd.DataFrame:
    def mk():
        r = open_service(REFDATA).createRequest("IntradayBarRequest")
        r.set("security", sec)
        r.set("eventType", event)
        r.set("interval", 1)
        r.set("startDateTime", start_dt)
        r.set("endDateTime", end_dt)
        r.set("gapFillInitialBar", False)
        return r

    rows, errs = parse_bars(send(mk), sec, event)
    for e in errs:
        log_error("bar_error", event=event, **e)
    df = pd.DataFrame(rows)
    if len(df):
        df["time"] = pd.to_datetime(df["time"], utc=True)
    return df


def ticks_request(sec: str, events, start_dt: datetime, end_dt: datetime) -> pd.DataFrame:
    def mk():
        r = open_service(REFDATA).createRequest("IntradayTickRequest")
        r.set("security", sec)
        for e in events:
            r.getElement("eventTypes").appendValue(e)
        r.set("startDateTime", start_dt)
        r.set("endDateTime", end_dt)
        r.set("includeConditionCodes", True)  # required: F1 tick cleaning keys on condition codes
        r.set("includeExchangeCodes", True)
        r.set("includeNonPlottableEvents", True)
        return r

    rows, errs = parse_ticks(send(mk, timeout=300), sec)
    for e in errs:
        log_error("tick_error", **e)
    df = pd.DataFrame(rows)
    if len(df):
        df["time"] = pd.to_datetime(df["time"], utc=True)
    return df

## 5. Library: universe, daily history, static data, events, probe

In [ ]:
EQUITY_DAILY = [  # the previous pull's validated set, minus fields Bloomberg rejected (DATA_AUDIT F9)
    "PX_OPEN",
    "PX_HIGH",
    "PX_LOW",
    "PX_LAST",
    "PX_BID",
    "PX_ASK",
    "PX_VOLUME",
    "TURNOVER",
    "EQY_WEIGHTED_AVG_PX",
    "NUM_TRADES",
    "EQY_SH_OUT",
    "CUR_MKT_CAP",
    "EQY_FREE_FLOAT_PCT",
    "EQY_DVD_YLD_IND",
    "TOT_RETURN_INDEX_GROSS_DVDS",
    "TOT_RETURN_INDEX_NET_DVDS",
    "DAY_TO_DAY_TOT_RETURN_GROSS_DVDS",
    "VOLATILITY_10D",
    "VOLATILITY_30D",
    "VOLATILITY_60D",
    "VOLATILITY_90D",
    "VOLATILITY_260D",
    "30DAY_IMPVOL_100.0%MNY_DF",
    "30DAY_IMPVOL_90.0%MNY_DF",
    "30DAY_IMPVOL_110.0%MNY_DF",
    "60DAY_IMPVOL_100.0%MNY_DF",
    "PUT_CALL_VOLUME_RATIO_CUR_DAY",
    "PUT_CALL_OPEN_INTEREST_RATIO",
    "PE_RATIO",
    "PX_TO_BOOK_RATIO",
    "VOLUME_AVG_30D",
    "PX_CLOSE_1D",
    "BETA_RAW_OVERRIDABLE",
]
RAW_PRICE_FIELDS = ["PX_OPEN", "PX_HIGH", "PX_LOW", "PX_LAST", "PX_VOLUME"]  # unadjusted copy (*_RAW)
HIST_ONLY_DAILY = ["PX_OPEN", "PX_HIGH", "PX_LOW", "PX_LAST", "PX_VOLUME", "TURNOVER", "CUR_MKT_CAP"]
DAILY_FIELDS = {
    "equity": EQUITY_DAILY,
    "equity_hist_only": HIST_ONLY_DAILY,
    "index": [
        "PX_OPEN",
        "PX_HIGH",
        "PX_LOW",
        "PX_LAST",
        "PX_VOLUME",
        "TURNOVER",
        "CUR_MKT_CAP",
        "TOT_RETURN_INDEX_GROSS_DVDS",
        "VOLATILITY_30D",
        "VOLATILITY_90D",
        "PE_RATIO",
        "PX_TO_BOOK_RATIO",
        "EQY_DVD_YLD_IND",
        "30DAY_IMPVOL_100.0%MNY_DF",
    ],
    "fx": ["PX_OPEN", "PX_HIGH", "PX_LOW", "PX_LAST", "PX_BID", "PX_ASK", "VOLATILITY_30D"],
    "rates": ["PX_OPEN", "PX_HIGH", "PX_LOW", "PX_LAST"],
    "commodity": ["PX_OPEN", "PX_HIGH", "PX_LOW", "PX_LAST", "PX_VOLUME", "OPEN_INT"],
    "futures": [
        "PX_OPEN",
        "PX_HIGH",
        "PX_LOW",
        "PX_LAST",
        "PX_SETTLE",
        "PX_BID",
        "PX_ASK",
        "PX_VOLUME",
        "OPEN_INT",
        "FUT_AGGTE_VOL",
        "FUT_AGGTE_OPEN_INT",
    ],
}
STATIC_FIELDS = {  # TICK_SIZE / TRADING_LOT_SIZE are not served (tick size is derived in hftf_data.derive)
    "equity": [
        "NAME",
        "SHORT_NAME",
        "SECURITY_TYP",
        "ID_BB_GLOBAL",
        "ID_ISIN",
        "ID_SEDOL1",
        "TICKER",
        "EXCH_CODE",
        "EQY_PRIM_EXCH",
        "CRNCY",
        "COUNTRY_ISO",
        "GICS_SECTOR_NAME",
        "GICS_INDUSTRY_GROUP_NAME",
        "GICS_INDUSTRY_NAME",
        "GICS_SUB_INDUSTRY_NAME",
        "INDUSTRY_SECTOR",
        "INDUSTRY_GROUP",
        "INDUSTRY_SUBGROUP",
        "EQY_SH_OUT",
        "EQY_FLOAT",
        "EQY_FREE_FLOAT_PCT",
        "CUR_MKT_CAP",
        "EQY_INIT_PO_DT",
        "ANNOUNCEMENT_DT",
        "EXPECTED_REPORT_DT",
        "LATEST_ANNOUNCEMENT_DT",
        "MARKET_STATUS",
        "PX_LAST",
        "VOLUME",
    ],
    "index": ["NAME", "ID_BB_GLOBAL", "COUNTRY_ISO", "CRNCY", "EXCH_CODE", "PX_LAST"],
    "fx": ["NAME", "ID_BB_GLOBAL", "PX_LAST"],
    "rates": ["NAME", "ID_BB_GLOBAL", "PX_LAST"],
    "commodity": ["NAME", "ID_BB_GLOBAL", "PX_LAST"],
    "futures": [
        "NAME",
        "ID_BB_GLOBAL",
        "FUT_CONT_SIZE",
        "FUT_TICK_SIZE",
        "LAST_TRADEABLE_DT",
        "FUT_FIRST_TRADE_DT",
        "OPEN_INT",
        "PX_LAST",
    ],
}
STATIC_FIELDS["equity_hist_only"] = STATIC_FIELDS["equity"]
EVENT_BULK_FIELDS = ["DVD_HIST_ALL", "EARN_ANN_DT_TIME_HIST_WITH_EPS"]


def _num(v) -> float:
    try:
        return float(v)
    except (TypeError, ValueError):
        return float("nan")


def resolve(secs) -> tuple:
    ok, bad, info = [], [], {}
    for batch in chunks(list(dict.fromkeys(secs)), 50):
        try:
            rows, _, _ = ref_request(batch, ["ID_BB_GLOBAL", "NAME", "CUR_MKT_CAP"])
        except BBGRequestFailure as e:
            log_error("resolve_failure", detail=str(e))
            rows = []
        good = {r["security"]: r for r in rows if r.get("ID_BB_GLOBAL")}
        for s in batch:
            (ok if s in good else bad).append(s)
            if s in good:
                info[s] = good[s]
    return ok, bad, info


def index_members(index: str) -> list:
    for fld in ("INDX_MWEIGHT", "INDX_MEMBERS"):  # INDX_MEMBERS = fallback without weights
        try:
            _, bulk, errs = ref_request([index], [fld])
        except BBGRequestFailure as e:
            log_error("members_failure", index=index, field=fld, detail=str(e))
            continue
        mem = [nse_equity(m) for m in (bulk_member(r)[0] for r in bulk) if m]
        if mem:
            return list(dict.fromkeys(mem))
    return []


def membership_history(index: str) -> pd.DataFrame:
    """Quarter-end point-in-time constituents (survivorship control). Fetches only quarters not on disk."""
    p = OUT / "reference" / "index_membership_history.parquet"
    old = pd.read_parquet(p) if p.exists() else pd.DataFrame(columns=["index", "asof", "member", "weight"])
    have = set(old["asof"].astype(str))
    qdates, d = [], date(DAILY_START.year, 3, 31)
    while d < TODAY:
        qdates.append(d)
        m = d.month + 3
        d = (pd.Timestamp(date(d.year + (m - 1) // 12, (m - 1) % 12 + 1, 1)) + pd.offsets.MonthEnd(0)).date()
    rows = []
    for qd in [q for q in qdates if str(q) not in have]:
        try:
            _, bulk, _ = ref_request(
                [index], ["INDX_MWEIGHT_HIST"], {"END_DATE_OVERRIDE": qd.strftime("%Y%m%d")}
            )
        except BBGRequestFailure as e:
            log_error("membership_failure", asof=str(qd), detail=str(e))
            continue
        for rec in bulk:
            mem, wt = bulk_member(rec)
            if mem:
                rows.append({"index": index, "asof": str(qd), "member": nse_equity(mem), "weight": wt})
    new = pd.DataFrame(rows)
    if len(new):
        out = pd.concat([old, new], ignore_index=True)
        out["asof"] = out["asof"].astype(str)
        out["member"] = out["member"].map(nse_equity)
        atomic_write(_stringify_nested(out.drop_duplicates(["index", "asof", "member"])), p)
        return out
    return old


def build_universe() -> dict:
    """Intraday equities = current Nifty 50 + names that already have bars (continuity) + a liquid buffer."""
    members = index_members(MEMBER_INDEX)
    if not members:
        log(f"!! {MEMBER_INDEX} membership returned nothing; using FALLBACK_EQUITIES")
        members = [nse_equity(s) for s in FALLBACK_EQUITIES]
    hist = membership_history(MEMBER_INDEX)
    continuity = equities_with_bars()
    buffer_cands = [s for s in index_members(BUFFER_INDEX) if s not in members] if BUFFER_N else []
    eq_ok, eq_bad, info = resolve(members + continuity + buffer_cands)
    cap = {s: _num(info[s].get("CUR_MKT_CAP")) for s in eq_ok}
    buffer = sorted((s for s in buffer_cands if s in eq_ok and cap[s] == cap[s]), key=lambda s: -cap[s])[
        :BUFFER_N
    ]
    equity = [s for s in dict.fromkeys(members + continuity + buffer) if s in eq_ok]
    ever = set(hist["member"]) if len(hist) else set()
    hist_only, hist_bad, _ = resolve(sorted(ever - set(equity))) if ever - set(equity) else ([], [], {})
    other = {
        "index": list(dict.fromkeys(CONTEXT_INDICES + DAILY_ONLY_INDICES + [BUFFER_INDEX])),
        "fx": FX_INTRADAY + FX_DAILY_ONLY,
        "rates": RATES,
        "commodity": COMMODITIES,
        "futures": FUTURES_INTRADAY + FUTURES_DAILY_ONLY,
    }
    uni = {"equity": equity, "equity_hist_only": hist_only}
    unresolved = eq_bad + hist_bad
    for cls, secs in other.items():
        ok, bad, _ = resolve(secs)
        uni[cls] = ok
        unresolved += bad
    uni["unresolved"] = sorted(set(unresolved))
    save_json(uni, OUT / "reference" / "universe.json")
    save_json(
        {
            "built_at": datetime.now(UTC).isoformat(),
            "members": len(members),
            "continuity_only": sorted(set(continuity) - set(members)),
            "buffer": buffer,
        },
        OUT / "reference" / "universe_meta.json",
    )
    log("universe: " + ", ".join(f"{k}={len(v)}" for k, v in uni.items()))
    log(
        f"  intraday equities: {len(members)} members + {len(set(continuity) - set(members))} continuity + "
        f"{len(buffer)} buffer {buffer}"
    )
    if uni["unresolved"]:
        log(f"  unresolved (check in the Terminal): {uni['unresolved']}")
    if len(equity) < 50:
        log(f"!! only {len(equity)} intraday equities: research-tier R01 needs >= 50")
    return uni


def load_universe() -> dict:
    uni = load_json(OUT / "reference" / "universe.json")
    if uni is None:
        eq = equities_with_bars()
        log(f"no universe.json yet; using the {len(eq)} equities already on disk")
        uni = {"equity": eq, "index": CONTEXT_INDICES, "fx": FX_INTRADAY, "futures": FUTURES_INTRADAY}
    uni["equity"] = [nse_equity(s) for s in uni.get("equity", [])]
    uni["futures"] = [f for f in uni.get("futures", []) if f in FUTURES_INTRADAY + FUTURES_DAILY_ONLY]
    return uni


# ---- daily history (incremental) -----------------------------------------------------------------
def daily_part(cls: str, sec: str) -> Path:
    return OUT / "daily" / "_parts" / f"{cls}__{dir_name(sec)}.parquet"


def equity_daily(cols: list) -> pd.DataFrame:
    """[security, date, *cols] for intraday equities, from the parts (Bloomberg PC) or daily_equity (Mac)."""
    parts = sorted((OUT / "daily" / "_parts").glob("equity__*.parquet"))
    if parts:
        frames = [pd.read_parquet(p, columns=["security", "date", *cols]) for p in parts]
        return pd.concat(frames, ignore_index=True)
    p = OUT / "daily" / "daily_equity.parquet"
    return (
        pd.read_parquet(p, columns=["security", "date", *cols])
        if p.exists()
        else pd.DataFrame(columns=["security", "date", *cols])
    )


def _fetch_daily(cls, sec, start, end) -> pd.DataFrame:
    df = hist_request(sec, DAILY_FIELDS[cls], start, end, adjust=True)
    if len(df) and cls == "equity":
        raw = hist_request(sec, RAW_PRICE_FIELDS, start, end, adjust=False)
        if len(raw):
            raw = raw.drop(columns=["security"]).rename(
                columns={c: f"{c}_RAW" for c in raw.columns if c != "date"}
            )
            df = df.merge(raw, on="date", how="left")
    return df


def daily_needs_update(cls: str, sec: str, end: date) -> bool:
    p = daily_part(cls, sec)
    if p.with_suffix(".empty").exists() and not RETRY_EMPTY:
        return False
    if not p.exists():
        return True
    return pd.to_datetime(pd.read_parquet(p, columns=["date"])["date"]).max().date() < end


def update_daily(cls: str, sec: str, end: date) -> str:
    p = daily_part(cls, sec)
    if not daily_needs_update(cls, sec, end):
        return "current"
    if p.exists():
        old = pd.read_parquet(p)
        old["date"] = pd.to_datetime(old["date"])
        new = _fetch_daily(cls, sec, old["date"].max().date() - timedelta(days=10), end)
        if new.empty:
            return "no new rows"
        both = old.merge(new[["date", "PX_LAST"]], on="date", suffixes=("", "_new"))
        same = (
            np.allclose(
                both["PX_LAST"].astype(float), both["PX_LAST_new"].astype(float), rtol=1e-4, equal_nan=True
            )
            if len(both)
            else False
        )
        if same:
            df = pd.concat([old, new], ignore_index=True).drop_duplicates("date", keep="last")
            status = "appended"
        else:  # split / adjustment changed the back history: the stored series is stale
            df = _fetch_daily(cls, sec, DAILY_START, end)
            status = "re-fetched (adjustment changed)"
    else:
        df = _fetch_daily(cls, sec, DAILY_START, end)
        status = "new"
        if df.empty:
            p.parent.mkdir(parents=True, exist_ok=True)
            p.with_suffix(".empty").write_text("")
            return "empty"
    df["class"] = cls
    atomic_write(coerce_numeric(df.sort_values("date").reset_index(drop=True)), p)
    return status


def collect_daily(uni: dict, end: date, classes=None) -> None:
    jobs = [(cls, s) for cls in DAILY_FIELDS for s in uni.get(cls, []) if not classes or cls in classes]
    todo = [(c, s) for c, s in jobs if daily_needs_update(c, s, end)]
    log(f"daily: {len(todo)}/{len(jobs)} securities need new rows (through {end})")
    counts: dict = {}
    for i, (cls, sec) in enumerate(todo, 1):
        st = update_daily(cls, sec, end)
        counts[st] = counts.get(st, 0) + 1
        if i % 20 == 0:
            log(f"  daily {i}/{len(todo)}")
    if counts:
        log(f"  daily results: {counts}")
    changed = any(k not in ("current", "no new rows", "empty") for k in counts)
    if changed or not (OUT / "daily" / "daily_index.parquet").exists():  # keeps the delta bundle small
        consolidate_daily()


def consolidate_daily() -> None:
    by_cls: dict = {}
    for p in sorted((OUT / "daily" / "_parts").glob("*.parquet")):
        cls = p.name.split("__")[0]
        by_cls.setdefault("equity" if cls == "equity_hist_only" else cls, []).append(p)
    for cls, ps in by_cls.items():
        frames = [f for f in (pd.read_parquet(p) for p in ps) if len(f)]
        if not frames:
            continue
        df = pd.concat(frames, ignore_index=True)
        df = coerce_numeric(
            df.sort_values(["security", "date"]).drop_duplicates(["security", "date"], keep="last")
        )
        atomic_write(df, OUT / "daily" / f"daily_{cls}.parquet")
        log(f"  daily_{cls}.parquet: {len(df):,} rows, {df['security'].nunique()} securities")


def collect_static(uni: dict) -> None:
    for cls, fields in STATIC_FIELDS.items():
        secs = uni.get(cls, [])
        if not secs:
            continue
        rows = []
        for batch in chunks(secs, 25):
            try:
                r, _, errs = ref_request(batch, fields)
            except BBGRequestFailure as e:
                log_error("static_failure", cls=cls, detail=str(e))
                continue
            rows += r
            for e in errs:
                log_error("static_error", **e)
        df = pd.DataFrame(rows)
        if len(df):
            df["retrieved_at"] = datetime.now(UTC).isoformat()
            # static_equity defines the intraday universe for hftf_data (readiness R01, build-features)
            atomic_write(_stringify_nested(df), OUT / "reference" / f"static_{cls}.parquet")
            log(f"  static_{cls}: {len(df)} securities")


def collect_events(uni: dict) -> None:
    acc: dict = {}
    for sec in uni.get("equity", []):
        try:
            _, bulk, errs = ref_request([sec], EVENT_BULK_FIELDS)
        except BBGRequestFailure as e:
            log_error("events_failure", security=sec, detail=str(e))
            continue
        for e in errs:
            log_error("events_error", **e)
        for rec in bulk:
            acc.setdefault(rec["field"], []).append(rec)
    for field, rows in acc.items():
        df = pd.DataFrame(rows).dropna(axis=1, how="all")
        atomic_write(_stringify_nested(df), OUT / "events" / f"{field}.parquet")
        log(f"  events/{field}: {len(df)} rows")


# ---- probe: timezone + real history depth --------------------------------------------------------
def _oldest(sessions: list, has) -> date | None:
    """Oldest session with data, by binary search (availability is a contiguous recent window)."""
    if not sessions or not has(sessions[-1]):
        return None
    lo, hi = 0, len(sessions) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if has(sessions[mid]):
            hi = mid
        else:
            lo = mid + 1
    return sessions[lo]


def probe(sample: str, end: date) -> dict:
    caps = {"sample_security": sample, "run_at": datetime.now(UTC).isoformat()}
    for svc in (APIFLDS, "//blp/mktdata", "//blp/mktdepthdata"):
        try:
            open_service(svc)
            caps[f"service_{svc}"] = True
        except Exception as e:
            caps[f"service_{svc}"] = False
            caps[f"service_{svc}_error"] = str(e)
    sess = nse_sessions(end - timedelta(days=400), end)
    df = bars_request(sample, "TRADE", utc_dt(sess[-1]), utc_dt(sess[-1], True))
    if df.empty:
        raise RuntimeError(
            f"probe: no 1-min bars for {sample} on {sess[-1]}. Check entitlement before pulling."
        )
    ist = to_ist_naive(df["time"])
    first, last = ist.min().time(), ist.max().time()
    caps["tz_check"] = {"date": str(sess[-1]), "first_bar_ist": str(first), "last_bar_ist": str(last)}
    if not (hhmm("08:45") <= first <= hhmm("09:30")):
        raise RuntimeError(
            f"probe: first bar at {first} after UTC->IST conversion (expected ~09:00-09:15). "
            "Bloomberg is not returning UTC on this terminal; fix before pulling."
        )
    log(f"probe {sample} {sess[-1]}: bars {first:%H:%M}-{last:%H:%M} IST (UTC handling OK)")

    def has_bars(d):
        return (
            len(
                bars_request(
                    sample,
                    "TRADE",
                    datetime(d.year, d.month, d.day, 3, 45),
                    datetime(d.year, d.month, d.day, 4, 0),
                )
            )
            > 0
        )

    def has_ticks(d):  # the morning must be there, so the oldest day found is a full day
        return (
            len(
                ticks_request(
                    sample,
                    ["TRADE"],
                    datetime(d.year, d.month, d.day, 3, 45),
                    datetime(d.year, d.month, d.day, 3, 50),
                )
            )
            > 0
        )

    ob = _oldest(sess, has_bars)
    ot = _oldest([d for d in sess if d >= end - timedelta(days=120)], has_ticks)
    caps["oldest_bar_session"] = str(ob) if ob else None
    caps["oldest_tick_session"] = str(ot) if ot else None
    caps["bar_lookback_days"] = (TODAY - ob).days if ob else 0
    caps["tick_lookback_days"] = (TODAY - ot).days if ot else 0
    save_json(caps, OUT / "reports" / "capabilities.json")
    log(
        f"probe: 1-min bars from {ob} ({caps['bar_lookback_days']}d), "
        f"ticks from {ot} ({caps['tick_lookback_days']}d)"
    )
    return caps


def history_windows(end: date) -> tuple:
    caps = load_json(OUT / "reports" / "capabilities.json", {}) or {}
    ob, ot = caps.get("oldest_bar_session"), caps.get("oldest_tick_session")
    bar_start = date.fromisoformat(ob) if ob else end - timedelta(days=DEFAULT_BAR_LOOKBACK_DAYS)
    tick_start = date.fromisoformat(ot) if ot else end - timedelta(days=DEFAULT_TICK_LOOKBACK_DAYS)
    return bar_start, tick_start

## 6. Library: gap planning, bars, ticks
A **series** is one (security, event). Its *gaps* are the NSE sessions inside Bloomberg's window that are missing
on disk or end before 15:20 IST. Consecutive gaps are requested together (up to 5 sessions per request). New files
are named `<first>_<last>__p<run>.parquet`, so they never overwrite older chunks. The loaders de-duplicate by
timestamp, keeping the newest file's rows.

In [ ]:
def bar_series(uni: dict) -> list:
    out = [("equity", s, ev) for s in uni.get("equity", []) for ev in ("TRADE", "BID", "ASK")]
    out += [
        ("futures", s, ev)
        for s in uni.get("futures", [])
        if s in FUTURES_INTRADAY
        for ev in ("TRADE", "BID", "ASK")
    ]
    out += [("index", s, "TRADE") for s in CONTEXT_INDICES if s in uni.get("index", CONTEXT_INDICES)]
    out += [("fx", s, ev) for s in uni.get("fx", []) if s in FX_INTRADAY for ev in ("BID", "ASK")]
    return out


def _runs(days: list, sessions: list, max_len: int) -> list:
    pos = {d: i for i, d in enumerate(sessions)}
    runs: list = []
    for d in days:
        if runs and pos[d] == pos[runs[-1][-1]] + 1 and len(runs[-1]) < max_len:
            runs[-1].append(d)
        else:
            runs.append([d])
    return runs


def plan_bars(uni: dict, start: date, end: date, ledger: Ledger) -> list:
    sessions = nse_sessions(start, end)
    full = hhmm(FULL_DAY_LAST_IST)
    plan = []
    for cls, sec, ev in bar_series(uni):
        have = day_extents(sorted(bar_folder(ev, sec).glob("*.parquet")))
        key = f"bars/{ev}/{dir_name(sec)}"
        missing = [d for d in sessions if d not in have and not ledger.get(key, d)]
        partial = [d for d in sessions if d in have and have[d][1] < full and not ledger.get(key, d)]
        gaps = sorted(missing + partial)
        plan.append(
            {
                "class": cls,
                "security": sec,
                "event": ev,
                "have": len(have),
                "missing": missing,
                "partial": partial,
                "runs": _runs(gaps, sessions, BAR_CHUNK_SESSIONS),
            }
        )
    return plan


def collect_bars(plan: list, ledger: Ledger) -> None:
    jobs = [p for p in plan if p["runs"]]
    n_req = sum(len(p["runs"]) for p in jobs)
    log(f"bars: {n_req} requests for {len(jobs)} series")
    full = hhmm(FULL_DAY_LAST_IST)
    done = 0
    for p in jobs:
        sec, ev, key = p["security"], p["event"], f"bars/{p['event']}/{dir_name(p['security'])}"
        for run in p["runs"]:
            a, b = run[0], run[-1]
            try:
                df = bars_request(sec, ev, utc_dt(a), utc_dt(b, True))
            except BBGRequestFailure as e:
                log_error("bars_failure", security=sec, event=ev, chunk=f"{a}..{b}", detail=str(e))
                continue
            got = {}
            if len(df):
                df["class"] = p["class"]
                atomic_write(df, bar_folder(ev, sec) / f"{a:%Y%m%d}_{b:%Y%m%d}__p{RUN_TAG}.parquet")
                ist = to_ist_naive(df["time"])
                got = ist.groupby(ist.dt.date).max().map(lambda t: t.time()).to_dict()
            for d in run:
                if d not in got:
                    ledger.mark(key, d, "empty")
                elif got[d] < full:
                    ledger.mark(key, d, f"partial_final {got[d]}")  # e.g. a special/short session
            done += 1
            if done % 50 == 0:
                ledger.save()
                log(f"  bars {done}/{n_req} requests")
    ledger.save()


def tick_universe(uni: dict, end: date) -> list:
    """Fixed tick names: names that already have ticks first (accumulation), then the most liquid."""
    p = OUT / "reference" / "tick_universe.json"
    saved = load_json(p, {}) or {}
    names = list(dict.fromkeys([nse_equity(s) for s in saved.get("securities", [])] + equities_with_ticks()))
    if len(names) < TICK_UNIVERSE_N:
        d = equity_daily(["TURNOVER"])
        turnover = (
            (
                d.sort_values("date")
                .groupby("security")
                .tail(60)
                .groupby("security")["TURNOVER"]
                .median()
                .dropna()
                .to_dict()
            )
            if len(d)
            else {}
        )
        ranked = sorted(
            (s for s in uni.get("equity", []) if s not in names), key=lambda s: -turnover.get(s, -1.0)
        )
        names += ranked[: TICK_UNIVERSE_N - len(names)]
    if MODE == "pull":
        save_json(
            {
                "securities": names,
                "updated_at": datetime.now(UTC).isoformat(),
                "rule": "existing tick names first, then 60-session median TURNOVER",
            },
            p,
        )
    return names


def tick_day_extent(path: Path):
    if not path.exists():
        return None
    return day_extents([path]).get(datetime.strptime(path.stem, "%Y%m%d").date())


def tick_day_full(ext) -> bool:
    return bool(ext) and ext[0] <= hhmm(FULL_TICK_DAY_FIRST_IST) and ext[1] >= hhmm(FULL_DAY_LAST_IST)


def plan_ticks(names: list, start: date, end: date, ledger: Ledger) -> list:
    sessions = nse_sessions(start, end)
    plan = []
    for sec in names:
        key = f"ticks/{dir_name(sec)}"
        todo, full_days = [], 0
        for d in sessions:
            ext = tick_day_extent(tick_folder(sec) / f"{d:%Y%m%d}.parquet")
            if tick_day_full(ext):
                full_days += 1
            elif not ledger.get(key, d):
                todo.append((d, ext))
        plan.append({"security": sec, "full_days_in_window": full_days, "todo": todo})
    return plan


def collect_ticks(plan: list, ledger: Ledger) -> None:
    n = sum(len(p["todo"]) for p in plan)
    log(f"ticks: {n} security-days for {len(plan)} securities")
    (h0, m0), (h1, m1) = TICK_WINDOW_UTC
    done = 0
    for p in plan:
        sec, key = p["security"], f"ticks/{dir_name(p['security'])}"
        for d, _old_ext in p["todo"]:
            path = tick_folder(sec) / f"{d:%Y%m%d}.parquet"
            try:
                df = ticks_request(
                    sec,
                    TICK_EVENTS,
                    datetime(d.year, d.month, d.day, h0, m0),
                    datetime(d.year, d.month, d.day, h1, m1),
                )
            except BBGRequestFailure as e:
                log_error("ticks_failure", security=sec, date=str(d), detail=str(e))
                continue
            done += 1
            if df.empty:
                ledger.mark(key, d, "empty")
                continue
            ist = to_ist_naive(df["time"])
            new_ext = (ist.min().time(), ist.max().time())
            old_rows = len(pd.read_parquet(path, columns=["time"])) if path.exists() else 0
            if len(df) > old_rows:  # never replace a day with less data
                atomic_write(_stringify_nested(df), path)
            if not tick_day_full(new_ext):
                ledger.mark(key, d, f"partial_final {new_ext[0]}-{new_ext[1]}")
            if done % 25 == 0:
                ledger.save()
                log(f"  ticks {done}/{n}")
        log(f"  ticks done: {sec}")
    ledger.save()

## 7. Library: readiness preview and bundle
This is a quick check on the Bloomberg PC, so you can see the gaps before leaving the Terminal. The binding gate
is still `uv run hftf data audit --tier research` on the Mac, which also checks tick cleaning, the imbalance
proxy, tick size and feature coverage.

In [ ]:
READINESS = {  # mirror of configs/data_readiness.yaml (coverage checks only)
    "pilot": {"eq": 3, "sessions": 60, "missing": 1, "tick_days": 15, "tick_eq": 3, "daily": 250},
    "research": {"eq": 50, "sessions": 90, "missing": 0, "tick_days": 40, "tick_eq": 10, "daily": 250},
}


def readiness_preview(uni: dict) -> dict:
    eq = uni.get("equity", [])
    cal = set(nse_sessions(date(2000, 1, 1), last_complete_session_day()))
    series = [(s, ev) for s in eq for ev in ("TRADE", "BID", "ASK")] + [(s, "TRADE") for s in CONTEXT_INDICES]
    sess, miss, miss_dates = {}, {}, set()
    for s, ev in series:
        days = set(day_extents(sorted(bar_folder(ev, s).glob("*.parquet"))))
        sess[(s, ev)] = len(days)
        if days:
            gap = {d for d in cal if min(days) <= d <= max(days)} - days
            miss[(s, ev)] = len(gap)
            miss_dates |= gap
    eq_with_bars = [s for s in eq if sess.get((s, "TRADE"))]
    full_ticks = {}
    for s in equities_with_ticks():
        full_ticks[s] = sum(tick_day_full(tick_day_extent(f)) for f in tick_folder(s).glob("*.parquet"))
    dd = equity_daily([])
    n_daily = dd.groupby("security")["date"].nunique().to_dict() if len(dd) else {}
    daily = {s: int(n_daily.get(s, 0)) for s in eq}
    min_sess = min((sess[(s, ev)] for s in eq for ev in ("TRADE", "BID", "ASK")), default=0)
    max_miss = max(miss.values(), default=0)
    rows = []
    for tier, th in READINESS.items():
        tick_ok = sum(v >= th["tick_days"] for v in full_ticks.values())
        rows += [
            (
                tier,
                "R01 intraday equities",
                len(eq_with_bars),
                f">= {th['eq']}",
                len(eq_with_bars) >= th["eq"],
            ),
            (
                tier,
                "R02 min sessions/equity/event",
                min_sess,
                f">= {th['sessions']}",
                min_sess >= th["sessions"],
            ),
            (tier, "R03 max missing sessions", max_miss, f"<= {th['missing']}", max_miss <= th["missing"]),
            (
                tier,
                "R04 context bars",
                sum(bool(sess.get((s, "TRADE"))) for s in CONTEXT_INDICES),
                "3",
                all(sess.get((s, "TRADE")) for s in CONTEXT_INDICES),
            ),
            (
                tier,
                f"R05 equities with >= {th['tick_days']} full tick days",
                tick_ok,
                f">= {th['tick_eq']}",
                tick_ok >= th["tick_eq"],
            ),
            (
                tier,
                "R10 min daily sessions/equity",
                min(daily.values(), default=0),
                f">= {th['daily']}",
                min(daily.values(), default=0) >= th["daily"],
            ),
        ]
    table = pd.DataFrame(rows, columns=["tier", "check", "measured", "required", "pass"])
    weak = sorted((f"{s}/{ev}={n}" for (s, ev), n in sess.items() if n < READINESS["research"]["sessions"]))
    out = {
        "generated_at": datetime.now(UTC).isoformat(),
        "checks": table.to_dict("records"),
        "missing_session_dates": sorted(str(d) for d in miss_dates),
        "full_tick_days": full_ticks,
        "series_below_90_sessions": weak,
    }
    print(table.to_string(index=False))
    if miss_dates:
        print("missing sessions inside bar windows:", ", ".join(sorted(str(d) for d in miss_dates))[:600])
    if weak:
        print(f"{len(weak)} series below 90 sessions:", ", ".join(weak[:15]), "..." if len(weak) > 15 else "")
    print(
        "full tick days:",
        ", ".join(f"{dir_name(k)}={v}" for k, v in sorted(full_ticks.items(), key=lambda kv: -kv[1])),
    )
    if MODE == "pull":
        save_json(out, OUT / "reports" / "pull_summary.json")
    return out


def bundle(since: datetime) -> Path | None:
    """Zip only the files written by this run. Unzip it INTO data/is2/ on the Mac (merge, never replace)."""
    files = [
        p
        for p in OUT.rglob("*")
        if p.is_file()
        and "_parts" not in p.relative_to(OUT).parts
        and not p.name.endswith((".empty", ".tmp"))
        and p.stat().st_mtime >= since.timestamp() - 1
    ]
    if not files:
        log("bundle: nothing new")
        return None
    path = OUT.parent / f"bbg_data_delta_{RUN_TAG}.zip"
    with zipfile.ZipFile(path, "w", compression=zipfile.ZIP_DEFLATED, allowZip64=True) as z:
        for p in files:
            z.write(p, Path("bbg_data") / p.relative_to(OUT))
    log(f"bundle: {len(files)} files -> {path} ({path.stat().st_size / 1e6:,.1f} MB)")
    return path

## 8. Run
`plan` prints the gap report and the request count. `pull` runs these steps in order: universe → daily history
(the session calendar comes from NIFTY daily) → probe → static/events → bars → ticks → readiness preview → bundle.
Every step is resumable. A data-limit stop skips the remaining Bloomberg steps but still writes the preview and
the bundle.

In [ ]:
def print_plan(bar_plan: list, tick_plan: list, bar_start: date, tick_start: date, end: date) -> None:
    n_bar_req = sum(len(p["runs"]) for p in bar_plan)
    n_missing = sum(len(p["missing"]) for p in bar_plan)
    n_partial = sum(len(p["partial"]) for p in bar_plan)
    new_series = sum(p["have"] == 0 for p in bar_plan)
    n_tick_req = sum(len(p["todo"]) for p in tick_plan)
    log(
        f"bar window {bar_start}..{end}: {len(bar_plan)} series ({new_series} new), "
        f"{n_missing} missing + {n_partial} partial sessions -> {n_bar_req} requests"
    )
    for p in bar_plan:
        if p["have"] and (p["missing"] or p["partial"]):
            log(
                f"  {dir_name(p['security'])}/{p['event']}: missing {[str(d) for d in p['missing']][:6]} "
                f"partial {[str(d) for d in p['partial']][:6]}"
            )
    log(f"tick window {tick_start}..{end}: {len(tick_plan)} securities -> {n_tick_req} security-day requests")
    for p in tick_plan:
        log(
            f"  {dir_name(p['security'])}: {p['full_days_in_window']} full days in window, "
            f"{len(p['todo'])} to fetch"
        )


def run(mode: str = MODE) -> dict:
    end = last_complete_session_day()
    ledger = Ledger(OUT / "reports" / "pull_ledger.json")
    log(f"mode={mode}  out={OUT.resolve()}  last complete session <= {end}")
    if mode == "plan":
        uni = load_universe()
        bar_start, tick_start = history_windows(end)
        bars_p = plan_bars(uni, bar_start, end, ledger)
        ticks_p = plan_ticks(tick_universe(uni, end), tick_start, end, ledger)
        print_plan(bars_p, ticks_p, bar_start, tick_start, end)
        return readiness_preview(uni)
    if mode != "pull":
        raise ValueError("MODE must be 'plan' or 'pull'")
    OUT.mkdir(parents=True, exist_ok=True)
    connect()
    uni, stopped = None, None
    steps = [
        ("universe", lambda: build_universe()),
        ("calendar", lambda: collect_daily(uni, end, classes=["index"])),
        (
            "probe",
            lambda: probe(
                next((s for s in uni["equity"] if s.startswith("RELIANCE")), uni["equity"][0]), end
            ),
        ),
        ("daily", lambda: collect_daily(uni, end)),
        ("static", lambda: collect_static(uni)),
        ("events", lambda: collect_events(uni)),
        (
            "plan",
            lambda: print_plan(
                plan_bars(uni, history_windows(end)[0], end, ledger),
                plan_ticks(tick_universe(uni, end), history_windows(end)[1], end, ledger),
                *history_windows(end),
                end,
            ),
        ),
        ("bars", lambda: collect_bars(plan_bars(uni, history_windows(end)[0], end, ledger), ledger)),
        (
            "ticks",
            lambda: collect_ticks(
                plan_ticks(tick_universe(uni, end), history_windows(end)[1], end, ledger), ledger
            ),
        ),
    ]
    for name, fn in steps:
        t0 = time.time()
        log(f"=== {name} ===")
        try:
            res = fn()
            if name == "universe":
                uni = res
            log(
                f"--- {name} finished in {(time.time() - t0) / 60:.1f} min "
                f"({STATS['requests']} requests so far)"
            )
        except BBGLimitReached as e:
            stopped = f"data limit during '{name}': {e}"
            log(f"!! Bloomberg data limit reached during '{name}'. Stopping; re-run tomorrow (gaps resume).")
            log_error("limit_reached", step=name, detail=str(e))
            break
        except Exception as e:
            log(f"!! step {name} failed: {e!r}")
            log_error("step_failed", step=name, detail=traceback.format_exc()[-1500:])
            if name in ("universe", "calendar", "probe"):
                stopped = f"{name} failed: {e!r}"
                break
    ledger.save()
    close()
    log(f"=== readiness preview ===  ({STATS['requests']} requests, {STATS['written_files']} files written)")
    summary = readiness_preview(uni or load_universe())
    summary["stopped"] = stopped
    save_json(summary, OUT / "reports" / "pull_summary.json")
    if MAKE_BUNDLE and not IN_REPO:
        bundle(RUN_STARTED)
    return summary


result = run(MODE)

## 9. After the pull (on the Mac)
```bash
unzip -o bbg_data_delta_<run>.zip -d data/is2/      # merges into data/is2/bbg_data; nothing is deleted
uv run hftf data build-features
uv run hftf data audit --tier research --strict
```
- **Weekly:** re-run this notebook with `MODE = "pull"` on the same Bloomberg PC folder. Each run fetches only new
  and missing sessions. R05 (≥ 40 full tick days on 10 names) needs ticks from at least two pulls about 4 weeks
  apart, because Bloomberg serves only about one month of ticks.
- **Bars roll off after about 140 days.** Run at least monthly, or sessions are lost for good. The 2026-09-10 gap
  can only be re-fetched while it is inside the window, i.e. before about late January 2027.
- If a new tick condition code or data quirk appears (R06 fails), record it in `docs/DATA_AUDIT.md`.